In [1]:
import numpy as np

In [3]:
def parse_bed_file(bedfile) -> list[tuple[int, int]]:
    """Parse a bed file and return a list of coordinates of 
       CpG islands in a genomic region."""
    
    cpg_islands = []

    with open(bedfile, "r") as f:
        for line in f:
            line = line.strip().split("\t")
            start = int(line[1])
            end = int(line[2])
            cpg_islands.append((start, end))

    return cpg_islands

#### The file ```human_hg38_chrX.bed``` contains the start and end coordinates of every annotated CpG insland in the raw DNA sequence in ```human_hg38_chrX.fasta```.

In [10]:
training_bedfile = "human_hg38_chrX.bed"
cpg_coords = parse_bed_file(training_bedfile)
print(cpg_coords)

[(249837, 251054), (253747, 254019), (254992, 255200), (255951, 256248), (261226, 261433), (267814, 268030), (275711, 277109), (280482, 281165), (288655, 288886), (290762, 291000), (293041, 293283), (299047, 299365)]


In [13]:
def pull_data(fasta_file, cpg_coords, c_start, c_end):
    """Parse a fasta file, annotate bases that belong to a CpG island as lower case
       and non-CpG regions as upper case. The function returns a sequence of bases in 
       upper case and the annotated hidden states of each base."""
       
    raw = "".join(line.strip() for line in open(fasta_file) if not line.startswith(">"))

    annotated_sequence = [b.upper() for b in raw]

    # Convert bases in cpg_coords to lowercase
    for start, end in cpg_coords:

        start_idx = start - c_start + 1
        end_idx = end - c_start + 1

        annotated_sequence[start_idx:end_idx] = [b.lower() for b in annotated_sequence[start_idx:end_idx]]

    sequence = [b.upper() for b in raw]

    hidden_states = ["CpG" if b.islower() else "non-CpG" for b in annotated_sequence] 
    return sequence, hidden_states 



#### The training sequence data ```human_hg38_chrX.fasta``` is a genomic region obtained from human reference genome hg38 assembly. The regions spans from base 240000 - 300000 in the X chromosome.

In [14]:

chrX_start = 240000
chrX_end = 300000

training_sequence = "human_hg38_chrX.fasta"
sequence, hidden_states = pull_data(training_sequence, cpg_coords, chrX_start, chrX_end)

# Number of CpG counts in the training data.
print(hidden_states.count("CpG"))

5527


In [17]:
def matrix_normalization(matrix):
    """Normalize a matrix across row"""
    
    return matrix / matrix.sum(axis=1, keepdims=True)


### Getting transition probability and store in a matrix: 

In [18]:
states = ["CpG", "non-CpG"]

# Initialize a 2 x 2 matrix to store transition probabilities of different states
transition_matrix = np.zeros((len(states), len(states)))

for current_state, next_state in zip(hidden_states[:-1], hidden_states[1:]):
    transition_matrix[states.index(current_state), states.index(next_state)] += 1


transition_prob = matrix_normalization(transition_matrix)
print(transition_prob)

[[9.97828840e-01 2.17115976e-03]
 [2.20292622e-04 9.99779707e-01]]


### Getting the emission probability and store in a matrix:

In [19]:
bases = ['A', 'T', 'G', 'C']

cpg_transition, noncpg_transition = np.zeros((len(bases), len(bases))), np.zeros((len(bases), len(bases)))

for i in range(len(sequence) - 1):

    current_base = sequence[i]
    next_base = sequence[i+1]

    if hidden_states [i+1] == "non-CpG":
        noncpg_transition[bases.index(current_base), bases.index(next_base)] += 1

    elif hidden_states [i+1] == "CpG":
        cpg_transition[bases.index(current_base), bases.index(next_base)] += 1

cpg_emission_prob, noncpg_emission_prob = matrix_normalization(cpg_transition), matrix_normalization(noncpg_transition)

emission_prob = { "CpG": cpg_emission_prob, "non-CpG": noncpg_emission_prob}


In [20]:
print(cpg_emission_prob)

[[0.15679825 0.0745614  0.51315789 0.25548246]
 [0.04545455 0.12899263 0.44963145 0.37592138]
 [0.18728374 0.15527682 0.44939446 0.20804498]
 [0.20013432 0.18737408 0.30087307 0.31161854]]


In [22]:
print(noncpg_emission_prob)

[[0.19642451 0.20922657 0.35164003 0.24270889]
 [0.17760144 0.24517343 0.32403469 0.25319045]
 [0.25597597 0.1781286  0.3576633  0.20823214]
 [0.33077153 0.2746605  0.1072228  0.28734517]]


### Implementing Viterbi Algorithmn to predict best path:

In [23]:
def viterbi(seq, states, initial_prob, transition_prob = transition_prob, emission_prob = emission_prob, bases = bases):
    
    num_row    = len(states)
    num_column = len(seq) 

    # Initialize a matrix to track the maximum probability of reaching a hidden state at each position
    bestpathprob = np.zeros((num_row, num_column))

    # Initialize a matrix to track...
    bestpathpointer = np.zeros((num_row, num_column), dtype=int)

    # Step 1: Initialization 
    for idx, state in enumerate(states):
        bestpathprob[idx, 0] = initial_prob[state]

    # We start iterating over the remaining sequences (start at index 1 since we have initialized
    # index 0 at step 1 
    for t in range(1, num_column):

        prev_base = bases.index(seq[t-1])
        curr_base = bases.index(seq[t])


        for curr_idx, curr_state in enumerate(states): 
            
            prob_history = []

            for prev_idx, prev_state in enumerate(states):
                curr_prob = ( np.log(transition_prob[prev_idx, curr_idx])
                            + np.log(emission_prob[curr_state][prev_base, curr_base]))
                
                prev_prob = bestpathprob[prev_idx, t-1]
                
                prob_history.append(curr_prob + prev_prob)
                
            bestpathprob[curr_idx, t] = max(prob_history)

            # np.argmax will return the index of the maximum value in the NumPy array.
            # We can then use the index to find the state.
            bestpathpointer[curr_idx, t] = np.argmax(prob_history)
                            

    # Termination steps: 
    # Find the maximum probability from the final stage
    bestprob_last = np.argmax(bestpathprob[:, -1])
    bestprob = np.max(bestpathprob[:, -1])


    # Reconstruct the path leading to the final state
    path = [states[bestprob_last]] 
    for i in range(num_column-1, 0, -1): 

        backpointer = bestpathpointer[bestprob_last, i]
        path.append(states[backpointer])

        bestprob_last = backpointer
       
         
    bestpath = path[::-1]
    

    return bestpath, bestprob


### Testing the algorithm on a DNA sequence (observable) from a different genomic region:

In [25]:
testing_bedfile = "test.bed"
test_cpg_coords =  parse_bed_file("test.bed")

In [28]:
testing_data = "test_data.fasta"
test_start = 253000
test_end = 257000

obs, obs_hidden = pull_data(testing_data, test_cpg_coords, test_start, test_end)

In [29]:
# Apply obs to Viterbi Algorithm
initial_probability = {"CpG":0, "non-CpG":1}
bestpath, prob = viterbi(obs, states, initial_probability)
print(bestpath.count("CpG"))

1615


In [34]:
# This block of code was copied and pasted from ChatGPT, i wanted 
# a quick and dirty way to check performance. Might change later
def find_cpg_regions(path):
    cpg_regions = []
    start = None

    for i, state in enumerate(path):
        if state == "CpG" and start is None:
            start = i

        elif state != "CpG" and start is not None:
            cpg_regions.append((start, i-1))
            start = None

    # Handle the case where the sequence ends in a CpG region
    if start is not None:
        cpg_regions.append((start, len(path)-1))

    return cpg_regions

In [31]:
predicted = find_cpg_regions(bestpath)
ground_true =  find_cpg_regions(obs_hidden)

In [32]:
print(predicted)

[(1044, 1266), (2609, 4000)]


In [33]:
print(ground_true)

[(935, 1266), (2765, 3865)]
